# Solving a Quadratic Unconstrained Binary Optimization instance

Solving a QUBO instance is straightforward with `qubo-solver`. We can directly use the `QuboSolver` class by providing a `QUBOInstance` with a given `SolverConfig` configuration.
`SolverConfig` specifies whether to use a classical approach or a quantum one. Note that `SolverConfig` comes with many options but the default ones can be used straightforwardly.
We have however more advanced tutorials on the quantum-related components to dive deeper into these advanced concepts.

## Solving with a quantum approach

To use a quantum approach, several choices have to be made regarging the configuration, explained in more details in the [`SolverConfig` section of the documentation](https://pasqal-io.github.io/qubo-solver/latest/content/solver/).

One main decision is about the [backend](https://pasqal-io.github.io/qubo-solver/latest/content/backend/), that is how we choose to perform quantum runs. We can decide to either perform our on emulators (locally, or remotely) or using a real quantum processing unit (QPU). Our QPU, based on the Rydberg Analog Model, is accessible remotely.

### Available backend types and devices

The supported backends are available via [`Qooqit`](https://pasqal-io.github.io/qoolqit/latest/api/qoolqit/execution/backends/), a Python package designed for algorithm development in the Rydberg Analog Model.

The backends can be divided into 3 main categories:

- [Local emulators](https://pasqal-io.github.io/emulators/latest/) (Qutip, Emu_mps, Emu_sv, ...),
- [Remote emulators]((https://docs.pasqal.com/cloud/emu-tn/)), which can be accessed via [`pasqal_cloud`](https://docs.pasqal.com/cloud/),
- [A remote QPU, such as Fresnel](https://docs.pasqal.com/cloud/fresnel-job/).

A backend will use device specifications to perform quantum computations. The list of supported devices can be found in [`the QoolQit devices documentation`](https://pasqal-io.github.io/qoolqit/latest/api/qoolqit/devices/).

### Running locally with an emulator

We can perform quantum simulations locally via an emulator (here, we choose the `QUTIP` emulator by default).

In [ ]:
import torch
from qubosolver import QUBOInstance
from qubosolver.config import SolverConfig, LocalEmulator
from qubosolver.solver import QuboSolver

# define QUBO
Q = torch.tensor([[-0.2, 0, 1.0], [0, 0,1.5], [1.0, 1.5, 0]])
instance = QUBOInstance(coefficients=Q)

# Create a SolverConfig object to use a quantum backend
config = SolverConfig(use_quantum=True, backend = LocalEmulator())

# Instantiate the quantum solver.
solver = QuboSolver(instance, config)

# Solve the QUBO problem.
solution = solver.solve()

# Display results
print(solution)

QUBOSolution(bitstrings=tensor([[1., 0., 0.],
        [0., 0., 1.],
        [0., 1., 0.],
        [0., 0., 0.]]), costs=tensor([-0.2000,  0.0000,  0.0000,  0.0000]), counts=tensor([30, 29, 31, 10]), probabilities=tensor([0.3000, 0.2900, 0.3100, 0.1000]), solution_status=<SolutionStatusType.UNPROCESSED: 'unprocessed'>)


### Running with a remote connection

We can decide to perform our runs remotely via [`pasqal_cloud`](https://docs.pasqal.com/cloud/).
To do so, we have to provide several information after [setting up an account](https://docs.pasqal.com/cloud/set-up/).

#### On a real QPU

The code above can be modified to solve the QUBO instance using our real QPU remotely as follows (run only with your `pasqal_cloud` information):

In [ ]:
import torch
from qubosolver import QUBOInstance
from qubosolver.config import SolverConfig
from qubosolver.solver import QuboSolver
from qubosolver.config import QPU, PasqalCloud

# Replace with your username, project id and password on the Pasqal Cloud.
USERNAME="#TO_PROVIDE"
PROJECT_ID="#TO_PROVIDE"
PASSWORD=None

if PASSWORD is not None:

    # define QUBO
    Q = torch.tensor([[-0.2, 0, 1.0], [0, 0,1.5], [1.0, 1.5, 0]])
    instance = QUBOInstance(coefficients=Q)

    connection = PasqalCloud(
        username=USERNAME,
        password=PASSWORD,
        project_id=PROJECT_ID,
    )
    config = SolverConfig(use_quantum=True, backend=QPU(connection=connection))
    # Run the solver
    solver = QuboSolver(instance, config)
    solutions = solver.solve()

    # Display results
    print(solutions)

#### On a remote emulators

Emulators are also available remotely via `pasqal_cloud`:

In [ ]:
import torch
from qubosolver import QUBOInstance
from qubosolver.config import SolverConfig
from qubosolver.solver import QuboSolver
from qubosolver.config import RemoteEmulator, PasqalCloud

# Replace with your username, project id and password on the Pasqal Cloud.
USERNAME="#TO_PROVIDE"
PROJECT_ID="#TO_PROVIDE"
PASSWORD=None

if PASSWORD is not None:

    # define QUBO
    Q = torch.tensor([[-0.2, 0, 1.0], [0, 0,1.5], [1.0, 1.5, 0]])
    instance = QUBOInstance(coefficients=Q)

    connection = PasqalCloud(
        username=USERNAME,
        password=PASSWORD,
        project_id=PROJECT_ID,
    )
    config = SolverConfig(use_quantum=True, backend=RemoteEmulator(connection=connection))
    # Run the solver
    solver = QuboSolver(instance, config)
    solutions = solver.solve()

    # Display results
    print(solutions)

## Solving with a classical approach

We show below an example of solving a QUBO using CPLEX.
More information on classical approaches can be found in the `Classical solvers` section of the `Contents` documentation.

In [ ]:
import torch
from qubosolver import QUBOInstance
from qubosolver.solver import QuboSolver
from qubosolver.config import ClassicalConfig, SolverConfig

# define QUBO
Q = torch.tensor([[-0.2, 0, 1.0], [0, 0,1.5], [1.0, 1.5, 0]])
instance = QUBOInstance(coefficients=Q)

# Create a SolverConfig object with classical solver options.
classical_config = ClassicalConfig(
    classical_solver_type="cplex",
    cplex_maxtime=10.0,
    cplex_log_path="test_solver.log",
)
config = SolverConfig(use_quantum=False, classical=classical_config)

# Instantiate the classical solver via the pipeline's classical solver dispatcher.
classical_solver = QuboSolver(instance, config)

# Solve the QUBO problem.
solution = classical_solver.solve()

# Display results
print(solution)

QUBOSolution(bitstrings=tensor([[1., 0., 0.]]), costs=tensor([-0.2000]), counts=None, probabilities=None, solution_status=<SolutionStatusType.UNPROCESSED: 'unprocessed'>)
